In [1]:
import numpy as np
import pandas as pd
import json
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')


class BrokerMatchingDataGenerator:
    def __init__(self, random_seed=42, observation_window_days=30,
                 initial_exploration_rate=0.15, exploration_decay=0.01):
        np.random.seed(random_seed)
        random.seed(random_seed)

        self.observation_window_days = observation_window_days
        self.initial_exploration_rate = initial_exploration_rate
        self.exploration_decay = exploration_decay
        self.reentry_rate = 0.08
        self.max_reentry_depth = 1
        self.global_time_step = 0

        # Conversion sigmoid parameters
        self.conversion_k = 5.0
        self.conversion_threshold = 0.5

        # Price sensitivity parameters (Ontario-specific base premia)
        self.price_sensitivity_k = 2.0
        self.expected_premium_base = {
            'auto': 1400,
            'home': 800,
            'bundle': 2000
        }

        # ── Seasonality (V7.2 crash fix: must be in __init__) ────────────────
        self.seasonality = {
            1: 0.9,  2: 0.95, 3: 1.0,  4: 1.05,
            5: 1.1,  6: 1.1,  7: 1.05, 8: 1.0,
            9: 1.05, 10: 1.0, 11: 0.95, 12: 0.85
        }

        # ── Market regime (V7.2 asymmetric transitions, V7.1 structure) ──────
        self.market_regime = 'normal'
        self.regime_counter = 0
        self.regime_durations = {'normal': 100, 'hard': 50, 'soft': 40}
        # V7.2 asymmetric transitions: hard market is stickier
        self.regime_transition = {
            'normal': {'hard': 0.3,  'soft': 0.3,  'normal': 0.4},
            'hard':   {'normal': 0.6, 'hard': 0.3,  'soft': 0.1},
            'soft':   {'normal': 0.7, 'soft': 0.2,  'hard': 0.1}
        }
        self.regime_effects = {'normal': 1.0, 'hard': 0.7, 'soft': 1.2}

        # ── Region definitions ────────────────────────────────────────────────
        self.regions = {
            'Toronto': 0.30,       'Mississauga': 0.10,  'Brampton': 0.07,
            'Ottawa': 0.09,        'Hamilton': 0.06,     'London': 0.05,
            'Markham': 0.05,       'Vaughan': 0.04,      'Kitchener': 0.04,
            'Windsor': 0.03,       'Oakville': 0.03,     'Burlington': 0.03,
            'Kingston': 0.02,      'Other_GTA': 0.05,    'Other_Southwestern': 0.04
        }

        self.nearby_regions = {
            'Toronto':      ['Mississauga', 'Markham', 'Vaughan', 'Oakville', 'Other_GTA'],
            'Mississauga':  ['Toronto', 'Brampton', 'Oakville', 'Burlington'],
            'Brampton':     ['Mississauga', 'Toronto', 'Vaughan'],
            'Ottawa':       ['Kingston'],
            'Hamilton':     ['Burlington', 'Oakville'],
            'London':       ['Kitchener', 'Windsor'],
            'Oakville':     ['Burlington', 'Mississauga', 'Toronto'],
            'Burlington':   ['Hamilton', 'Oakville', 'Mississauga'],
            'Kingston':     ['Ottawa']
        }

        self.insurance_types = ['auto', 'home', 'bundle']
        self.insurance_dist  = [0.45,  0.25,  0.30]
        self.expertise_areas = ['auto', 'home', 'commercial', 'bundle']

    # =========================================================================
    # UTILITY
    # =========================================================================

    def get_current_exploration_rate(self):
        """Decaying exploration rate — ensures propensity denominator shrinks
        toward exploit-only as the policy matures."""
        return max(0.05,
                   self.initial_exploration_rate
                   * np.exp(-self.global_time_step * self.exploration_decay))

    def update_market_regime(self):
        """Markov regime transition — called ONCE per lead (V7.1 fix)."""
        self.regime_counter += 1
        if self.regime_counter >= self.regime_durations[self.market_regime]:
            self.regime_counter = 0
            probs = self.regime_transition[self.market_regime]
            self.market_regime = np.random.choice(
                list(probs.keys()), p=list(probs.values())
            )

    def get_market_factor(self):
        return self.regime_effects[self.market_regime]

    # =========================================================================
    # BROKER GENERATION (V7.1 base + V7.2 ribo_licensed bool + patience fields)
    # =========================================================================

    def generate_brokers(self, n_brokers=300):
        """Generate broker profiles with RIBO licensing, burnout risk,
        efficiency, and career-stage-aware skill initialisation."""
        brokers = []

        region_list = []
        for region, weight in self.regions.items():
            region_list.extend([region] * int(n_brokers * weight))
        while len(region_list) < n_brokers:
            region_list.append(random.choice(list(self.regions.keys())))
        random.shuffle(region_list)

        for i in range(n_brokers):
            is_new = random.random() < 0.1

            # ── RIBO licensing (V7.2 cleaner boolean) ────────────────────────
            ribo_years   = np.random.randint(0, 2) if is_new else np.random.randint(0, 25)
            ribo_licensed = ribo_years >= 1          # simple, auditable rule

            # ── Skill (V7.1: influenced by RIBO years) ───────────────────────
            skill_base       = 0.50 if is_new else 0.65
            skill_ribo_bonus = 0.10 if ribo_licensed else 0.0
            skill_years_bonus = min(0.15, ribo_years * 0.01)
            skill = np.clip(
                np.random.normal(skill_base + skill_ribo_bonus + skill_years_bonus, 0.12),
                0.3, 0.9
            )

            n_expertise = np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2])
            expertise   = random.sample(self.expertise_areas, n_expertise)
            lang_choice = np.random.choice(
                ['English', 'French', 'Bilingual'], p=[0.85, 0.05, 0.10]
            )

            reliability   = np.clip(np.random.normal(0.85, 0.10), 0.50, 0.98)
            burnout_risk  = np.random.uniform(0.0, 0.30)
            commission_rate = np.random.uniform(0.08, 0.15)
            cost_per_lead   = np.random.uniform(50, 150)
            efficiency      = np.random.uniform(0.5, 1.5)

            brokers.append({
                'broker_id':            f'BR-{i+1:04d}',
                'region':               region_list[i],
                'expertise_auto':       int('auto'       in expertise),
                'expertise_home':       int('home'       in expertise),
                'expertise_commercial': int('commercial' in expertise),
                'expertise_bundle':     int('bundle'     in expertise),
                'languages':            lang_choice,
                'ribo_licensed':        ribo_licensed,
                'ribo_license_years':   ribo_years,
                'conversion_rate':      round(np.clip(skill * 0.4 + np.random.normal(0, 0.05), 0.05, 0.65), 3),
                'csat_score':           round(np.clip(np.random.normal(4.0 + skill * 0.5, 0.5), 2.5, 5.0), 2),
                'current_caseload':     np.random.randint(5, 40),
                'capacity':             np.random.choice([40, 50, 60, 75], p=[0.40, 0.30, 0.20, 0.10]),
                'avg_response_time':    round(np.random.uniform(2, 12), 1),
                'skill_level':          round(skill, 3),
                'learning_rate':        round(np.random.uniform(0.05, 0.25), 3),
                'burnout_risk':         round(burnout_risk, 3),
                'reliability':          round(reliability, 3),
                'commission_rate':      round(commission_rate, 3),
                'cost_per_lead':        round(cost_per_lead, 2),
                'efficiency':           round(efficiency, 3),
                'years_experience':     0 if is_new else np.random.randint(1, 25),
                'is_new_broker':        is_new,
                'active':               True
            })

        return pd.DataFrame(brokers)

    # ── Churn support (grafted from V7.2) ─────────────────────────────────────

    def _next_broker_id(self, brokers_df):
        max_num = max(int(b.split('-')[1]) for b in brokers_df['broker_id'])
        return f'BR-{max_num + 1:04d}'

    def create_replacement_broker(self, brokers_df):
        """Generate a fresh new-hire broker to replace a churned one."""
        new_id      = self._next_broker_id(brokers_df)
        region      = np.random.choice(list(self.regions.keys()),
                                       p=list(self.regions.values()))
        n_expertise = np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2])
        expertise   = random.sample(self.expertise_areas, n_expertise)
        lang        = np.random.choice(['English', 'French', 'Bilingual'],
                                       p=[0.85, 0.05, 0.10])
        ribo_years    = np.random.randint(0, 3)
        ribo_licensed = ribo_years >= 1
        skill = np.clip(np.random.normal(0.45, 0.12), 0.30, 0.80)

        return {
            'broker_id':            new_id,
            'region':               region,
            'expertise_auto':       int('auto'       in expertise),
            'expertise_home':       int('home'       in expertise),
            'expertise_commercial': int('commercial' in expertise),
            'expertise_bundle':     int('bundle'     in expertise),
            'languages':            lang,
            'ribo_licensed':        ribo_licensed,
            'ribo_license_years':   ribo_years,
            'conversion_rate':      round(np.clip(skill * 0.4 + np.random.normal(0, 0.05), 0.05, 0.55), 3),
            'csat_score':           round(np.clip(np.random.normal(3.8 + skill * 0.5, 0.5), 2.5, 4.8), 2),
            'current_caseload':     max(3, np.random.poisson(10)),
            'capacity':             np.random.choice([40, 50, 60], p=[0.5, 0.3, 0.2]),
            'avg_response_time':    round(np.random.uniform(3, 12), 1),
            'skill_level':          round(skill, 3),
            'learning_rate':        round(np.random.uniform(0.10, 0.30), 3),
            'burnout_risk':         round(np.random.uniform(0.0, 0.20), 3),
            'reliability':          round(np.random.uniform(0.75, 0.95), 3),
            'commission_rate':      round(np.random.uniform(0.08, 0.15), 3),
            'cost_per_lead':        round(np.random.uniform(50, 150), 2),
            'efficiency':           round(np.random.uniform(0.5, 1.5), 3),
            'years_experience':     0,
            'is_new_broker':        True,
            'active':               True
        }

    def handle_broker_churn(self, broker_state, brokers_df):
        """Retire high-burnout / over-capacity / senior brokers; spawn
        replacements so pool size stays roughly constant."""
        churned     = []
        replacements = []

        for broker_id, state in list(broker_state.items()):
            if not state.get('active', True):
                continue
            row = brokers_df[brokers_df['broker_id'] == broker_id]
            if row.empty:
                continue
            broker = row.iloc[0]

            churn_prob = 0.001                          # baseline
            if state['skill'] < 0.30:
                churn_prob += 0.020
            if state['caseload'] > broker['capacity'] * 1.2:
                churn_prob += 0.010
            if broker.get('years_experience', 0) > 30:
                churn_prob += 0.005
            if state.get('burnout_risk', 0) > 0.6:
                churn_prob += 0.015

            if random.random() < churn_prob:
                broker_state[broker_id]['active'] = False
                churned.append(broker_id)

                new_broker = self.create_replacement_broker(brokers_df)
                broker_state[new_broker['broker_id']] = {
                    'caseload':     new_broker['current_caseload'],
                    'skill':        new_broker['skill_level'],
                    'recent_success': 0.5,
                    'burnout_risk': new_broker['burnout_risk'],
                    'active':       True
                }
                replacements.append(new_broker)

        return broker_state, churned, replacements

    # =========================================================================
    # LEAD GENERATION (V7.1 base + V7.2 claims/multi-product/patience fields)
    # =========================================================================

    def generate_leads(self, n_leads=15000, start_date='2023-01-01'):
        leads = []
        start_date_obj = datetime.strptime(start_date, '%Y-%m-%d')
        end_date_obj   = datetime.now()
        date_range     = (end_date_obj - start_date_obj).days

        for i in range(n_leads):
            lead_date = start_date_obj + timedelta(days=np.random.randint(0, date_range))

            month          = lead_date.month
            season_factor  = self.seasonality[month]
            is_weekend     = lead_date.weekday() >= 5
            weekday_factor = 0.6 if is_weekend else 1.0

            if random.random() > (season_factor * weekday_factor * 0.8):
                continue

            region         = np.random.choice(list(self.regions.keys()),
                                              p=list(self.regions.values()))
            insurance_type = np.random.choice(self.insurance_types,
                                              p=self.insurance_dist)
            lead_language  = np.random.choice(['English', 'French'], p=[0.9, 0.1])

            hour_of_day = np.random.choice(
                [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
                p=[0.12, 0.13, 0.12, 0.10, 0.08, 0.07, 0.07, 0.08, 0.08, 0.07, 0.05, 0.03]
            )
            tenure = np.random.choice([0, 1, 2, 3, 4, 5], p=[0.4, 0.2, 0.15, 0.1, 0.08, 0.07])

            time_factor   = (lead_date - start_date_obj).days / date_range if date_range > 0 else 0
            digital_score = min(100, max(0, np.random.beta(2, 5) * 100 + 15 * time_factor))

            # Quote value anchored to insurance type (V7.1)
            base_quote   = self.expected_premium_base[insurance_type]
            quote_value  = round(base_quote * np.random.uniform(0.70, 1.50), 2)

            lead_difficulty    = round(np.random.uniform(0.7, 1.3), 3)
            sophistication     = round(np.random.uniform(0, 1), 3)

            # V7.2: patience_hours (exponential, capped)
            patience_hours = round(min(168, max(6, np.random.exponential(48))), 1)

            # V7.2: claims history (Ontario: ~30% have some claim in 5 yrs)
            claims_severity = np.random.choice(
                ['none', 'minor', 'major'], p=[0.70, 0.20, 0.10]
            )

            # V7.2: multi-product intent
            multi_product_intent = (
                insurance_type == 'bundle'
                or random.random() < 0.15
            )

            # 7% missing-data rate on key fields
            has_missing = random.random() < 0.07

            leads.append({
                'lead_id':                  f'LD-{i+1:06d}',
                'lead_date':                lead_date,
                'region':                   region,
                'postal_code_prefix':       region[:3].upper(),
                'insurance_type':           insurance_type   if not has_missing or random.random() > 0.3 else np.nan,
                'language':                 lead_language    if not has_missing or random.random() > 0.3 else np.nan,
                'tenure_years':             tenure           if not has_missing or random.random() > 0.2 else np.nan,
                'digital_engagement_score': round(digital_score, 1) if not has_missing or random.random() > 0.1 else np.nan,
                'quote_value':              quote_value,
                'lead_difficulty':          lead_difficulty,
                'sophistication':           sophistication,
                'patience_hours':           patience_hours,
                'claims_severity':          claims_severity,
                'multi_product_intent':     multi_product_intent,
                'hour_of_day':              hour_of_day,
                'is_weekend':               is_weekend,
                'month':                    month,
            })

        return pd.DataFrame(leads)

    # =========================================================================
    # MATCH SCORE  (V7.1 transparent — NO hidden factor)
    # =========================================================================

    def calculate_match_score(self, lead, broker):
        """Fully observable match score — hidden factor deliberately excluded
        so propensity scores computed from features are unbiased."""

        # 1. Expertise
        insurance_type = lead.get('insurance_type')
        expertise = 0.0
        if pd.notna(insurance_type):
            if insurance_type == 'auto'  and broker['expertise_auto']:  expertise = 1.0
            elif insurance_type == 'home' and broker['expertise_home']: expertise = 1.0
            elif insurance_type == 'bundle':
                if broker['expertise_bundle']:                          expertise = 1.0
                elif broker['expertise_auto'] and broker['expertise_home']: expertise = 0.9
                else:                                                   expertise = 0.6

        # 2. Language
        lang = lead.get('language')
        if pd.notna(lang):
            language = 1.0 if broker['languages'] in ('Bilingual', lang) else 0.3
        else:
            language = 0.8

        # 3. Workload
        workload = np.clip(1 - broker['current_caseload'] / broker['capacity'], 0.2, 1.0)

        # 4. Quality
        quality = np.clip(broker['conversion_rate'] * 0.6 + (broker['csat_score'] / 5) * 0.4, 0, 1)

        # 5. Response time
        response = max(0.5, min(1.0, 1 - broker['avg_response_time'] / 24))

        # Average of 5 components
        observed_score = (expertise + language + workload + quality + response) / 5

        # RIBO as clean ×1.1 multiplicative lift, capped at 1.0 (V7.1/V7.2 fix)
        if broker.get('ribo_licensed', False):
            observed_score = min(1.0, observed_score * 1.1)

        return np.clip(observed_score, 0.1, 1.0)

    # =========================================================================
    # PRICE SENSITIVITY  (V7.1, unchanged)
    # =========================================================================

    def calculate_price_sensitivity(self, quote_value, insurance_type, lead):
        expected = self.expected_premium_base.get(insurance_type,
                   self.expected_premium_base['auto'])
        if lead.get('tenure_years', 0) and lead['tenure_years'] > 0:
            expected *= 0.95           # existing customers expect slight discount
        price_ratio  = quote_value / expected
        price_effect = 1 / (1 + np.exp(self.price_sensitivity_k * (price_ratio - 1)))
        return np.clip(price_effect, 0.30, 1.20)

    # =========================================================================
    # SIGMOID CONVERSION  (V7.1 base + V7.2 claims_factor + multi_product_factor)
    # =========================================================================

    def sigmoid_conversion(self, base_score, lead, broker,
                           workload_ratio=0.5, quote_value=None):
        adjusted = base_score
        adjusted *= (1 - workload_ratio * 0.3)         # fatigue
        adjusted *= lead.get('lead_difficulty', 1.0)   # difficulty

        # Price sensitivity
        ins_type = lead.get('insurance_type', 'auto')
        if pd.notna(ins_type) and quote_value is not None:
            adjusted *= self.calculate_price_sensitivity(quote_value, ins_type, lead)

        # ── V7.2 graft: claims history ────────────────────────────────────────
        claims_severity = lead.get('claims_severity', 'none')
        if claims_severity == 'major':
            adjusted *= 0.85
        elif claims_severity == 'minor':
            adjusted *= 0.95

        # ── V7.2 graft: multi-product intent × broker bundle capability ───────
        if lead.get('multi_product_intent', False):
            if broker.get('expertise_bundle', 0):
                adjusted *= 1.15
            elif broker.get('expertise_auto', 0) and broker.get('expertise_home', 0):
                adjusted *= 1.05

        raw_prob = 1 / (1 + np.exp(-self.conversion_k * (adjusted - self.conversion_threshold)))
        return np.clip(raw_prob * 0.5, 0.02, 0.35)

    # =========================================================================
    # ELIGIBLE BROKER FILTERING
    # =========================================================================

    def get_eligible_brokers(self, lead, brokers_df):
        same_region = brokers_df[brokers_df['region'] == lead['region']]
        nearby = pd.DataFrame()
        if lead['region'] in self.nearby_regions and random.random() < 0.3:
            nearby = brokers_df[brokers_df['region'].isin(
                self.nearby_regions[lead['region']]
            )]

        if len(same_region) < 5:
            eligible = pd.concat([same_region, nearby])
        elif random.random() < 0.2 and len(nearby) > 0:
            eligible = pd.concat([same_region.sample(min(3, len(same_region))), nearby])
        else:
            eligible = same_region

        eligible = eligible[eligible['current_caseload'] <= eligible['capacity']]
        if len(eligible) == 0:
            eligible = brokers_df
        return eligible

    # =========================================================================
    # JOURNEY SIMULATION  (V7.1 propensity + V7.2 patience-aware interactions)
    # =========================================================================

    def simulate_journey(self, lead, brokers, broker_state, depth=0):
        assignments          = []
        counterfactual_samples = []
        attempted            = []
        current_exploration  = self.get_current_exploration_rate()

        # V7.2: patience drives max interactions (capped at 3)
        patience_hours  = lead.get('patience_hours', 48)
        max_interactions = min(3, int(patience_hours / 24) + 1)

        for interaction_num in range(max_interactions):

            # Refresh broker caseloads from live state each iteration (V7.1 fix)
            brokers_fresh = brokers.copy()
            brokers_fresh['current_caseload'] = brokers_fresh['broker_id'].map(
                lambda x: broker_state[x]['caseload']
            )

            available = brokers_fresh[~brokers_fresh['broker_id'].isin(attempted)]
            if available.empty:
                break

            # Score candidates with current skill (saturated learning curve)
            scores = []
            for _, b in available.iterrows():
                b_copy = b.copy()
                state  = broker_state[b['broker_id']]
                # saturated skill gain
                success_sig  = state.get('recent_success', 0.5)
                skill_gain   = b['learning_rate'] * (1 - state['skill']) * success_sig
                b_copy['skill_level'] = min(0.95, state['skill'] + skill_gain)
                # burnout degrades effective skill
                burnout = state.get('burnout_risk', 0)
                b_copy['skill_level'] *= (1 - burnout * 0.5)
                scores.append(self.calculate_match_score(lead, b_copy))

            available = available.copy()
            available['_score'] = scores
            available = available.sort_values('_score', ascending=False).head(10)

            # Softmax selection probabilities
            logits = np.exp(available['_score'] - available['_score'].max())
            probs  = logits / logits.sum()
            n_cand = len(available)

            action_probs = {
                str(bid): float(p)
                for bid, p in zip(available['broker_id'], probs)
            }

            # Mixed explore / exploit with unbiased propensity (V7.1)
            if random.random() < current_exploration:
                selected     = available.sample(1).iloc[0]
                sel_idx      = available.index.get_loc(selected.name)
                true_prop    = (current_exploration * (1 / n_cand)
                                + (1 - current_exploration) * probs.iloc[sel_idx])
            else:
                sel_idx      = np.random.choice(len(available), p=probs)
                selected     = available.iloc[sel_idx]
                true_prop    = (current_exploration * (1 / n_cand)
                                + (1 - current_exploration) * probs.iloc[sel_idx])

            attempted.append(selected['broker_id'])
            workload_ratio = selected['current_caseload'] / selected['capacity']

            # Burnout accumulates with workload
            broker_state[selected['broker_id']]['burnout_risk'] = min(0.80,
                broker_state[selected['broker_id']].get('burnout_risk', 0)
                + workload_ratio * 0.05
            )

            # Response check — burnout reduces effective reliability
            burnout_now     = broker_state[selected['broker_id']].get('burnout_risk', 0)
            eff_reliability = selected['reliability'] * (1 - burnout_now * 0.5) * selected['efficiency']

            if random.random() > eff_reliability:
                assignments.append({
                    'lead_id':                  lead['lead_id'],
                    'original_lead_id':         lead.get('original_lead_id', lead['lead_id']),
                    'broker_id':                selected['broker_id'],
                    'assignment_date':          lead['lead_date'],
                    'converted':                0,
                    'censored':                 0,
                    'conversion_delay_days':    np.nan,
                    'conversion_value':         0,
                    'revenue':                  0,
                    'cost':                     selected['cost_per_lead'],
                    'profit':                   -selected['cost_per_lead'],
                    'expected_profit':          -selected['cost_per_lead'],
                    'is_assigned':              1,
                    'interaction_number':       interaction_num,
                    'responded':                0,
                    'response_time_hours':      np.nan,
                    'propensity_score':         round(float(true_prop), 6),
                    'exploration_rate_at_time': round(current_exploration, 4),
                    'action_probabilities':     json.dumps(action_probs),
                    'position_bias':            0.0,
                    'market_regime':            self.market_regime,
                })
                continue

            # Response time with queue dynamics
            queue_mult    = 1 + workload_ratio * np.random.uniform(0.5, 1.5) / selected['efficiency']
            response_time = np.clip(
                np.random.exponential(selected['avg_response_time']) * queue_mult,
                0.5, 72
            )

            position_bias = float(np.exp(-0.3 * interaction_num))

            # Competition effect
            top_scores = available['_score'].head(3).values
            if len(top_scores) >= 2:
                competition_factor = 1 - np.clip(top_scores[0] - top_scores[1], 0, 0.3)
            else:
                competition_factor = 1.0

            # Full conversion probability chain
            base_score      = self.calculate_match_score(lead, selected)
            conversion_prob = self.sigmoid_conversion(
                base_score, lead, selected, workload_ratio, lead['quote_value']
            )
            conversion_prob *= position_bias
            conversion_prob *= competition_factor
            conversion_prob *= self.seasonality.get(lead.get('month', 6), 1.0)
            conversion_prob *= self.get_market_factor()
            conversion_prob  = np.clip(conversion_prob, 0.02, 0.35)

            will_convert = random.random() < conversion_prob

            revenue        = lead['quote_value'] * selected['commission_rate'] if will_convert else 0
            cost           = selected['cost_per_lead']
            profit         = revenue - cost if will_convert else -cost
            expected_profit = (lead['quote_value'] * selected['commission_rate'] * conversion_prob) - cost

            common = {
                'lead_id':                  lead['lead_id'],
                'original_lead_id':         lead.get('original_lead_id', lead['lead_id']),
                'broker_id':                selected['broker_id'],
                'assignment_date':          lead['lead_date'],
                'censored':                 0,
                'conversion_delay_days':    np.nan,
                'conversion_value':         0,
                'revenue':                  0,
                'cost':                     cost,
                'profit':                   -cost,
                'expected_profit':          round(expected_profit, 2),
                'is_assigned':              1,
                'interaction_number':       interaction_num,
                'responded':                1,
                'response_time_hours':      round(response_time, 1),
                'propensity_score':         round(float(true_prop), 6),
                'exploration_rate_at_time': round(current_exploration, 4),
                'action_probabilities':     json.dumps(action_probs),
                'position_bias':            round(position_bias, 4),
                'market_regime':            self.market_regime,
            }

            if will_convert:
                delay = np.clip(
                    np.random.exponential(3 / selected['efficiency']), 0.1, 30
                )
                observed = delay <= self.observation_window_days

                rec = {**common,
                       'converted':             int(observed),
                       'censored':               1 if (will_convert and not observed) else 0,
                       'conversion_delay_days':  round(delay, 1) if observed else np.nan,
                       'conversion_value':       lead['quote_value'] if observed else 0,
                       'revenue':                revenue if observed else 0,
                       'profit':                 profit  if observed else -cost}
                assignments.append(rec)

                # Skill update (saturated) + event-driven caseload -1
                success_sig = conversion_prob / 0.35
                state = broker_state[selected['broker_id']]
                state['skill'] = min(0.95,
                    state['skill'] + selected['learning_rate']
                    * (1 - state['skill']) * success_sig
                )
                state['caseload']       = max(0, state['caseload'] - 1)   # event-driven
                state['recent_success'] = success_sig
                state['burnout_risk']   = max(0, state.get('burnout_risk', 0) - 0.10)
                break

            else:
                assignments.append({**common, 'converted': 0})

                # event-driven: partial caseload release on timeout
                state = broker_state[selected['broker_id']]
                state['caseload']       = max(0, state['caseload'] - 0.5)
                state['recent_success'] = max(0.2,
                    state.get('recent_success', 0.5) * 0.95
                )

        # ── Uniform negative samples (V7.1 — no score-weighted leakage) ──────
        n_negatives = min(3, len(brokers) - len(attempted))
        if n_negatives > 0:
            unassigned = brokers[~brokers['broker_id'].isin(attempted)]
            if len(unassigned) > n_negatives:
                negatives = unassigned.sample(n=n_negatives)
            else:
                negatives = unassigned

            for _, neg in negatives.iterrows():
                would_prob = self.sigmoid_conversion(
                    self.calculate_match_score(lead, neg),
                    lead, neg, 0.5, lead['quote_value']
                )
                soft_convert    = random.random() < would_prob
                cf_revenue      = lead['quote_value'] * neg['commission_rate'] if soft_convert else 0
                cf_exp_profit   = lead['quote_value'] * neg['commission_rate'] * would_prob

                assignments.append({
                    'lead_id':                  lead['lead_id'],
                    'original_lead_id':         lead.get('original_lead_id', lead['lead_id']),
                    'broker_id':                neg['broker_id'],
                    'assignment_date':          lead['lead_date'],
                    'converted':                0,
                    'censored':                 0,
                    'conversion_delay_days':    np.nan,
                    'conversion_value':         0,
                    'revenue':                  0,
                    'cost':                     0,
                    'profit':                   0,
                    'expected_profit':          round(cf_exp_profit, 2),
                    'is_assigned':              0,
                    'interaction_number':       -1,
                    'responded':                np.nan,
                    'response_time_hours':      np.nan,
                    'propensity_score':         0.0,
                    'exploration_rate_at_time': round(current_exploration, 4),
                    'action_probabilities':     '{}',
                    'position_bias':            0.0,
                    'market_regime':            self.market_regime,
                })

                counterfactual_samples.append({
                    'lead_id':                        lead['lead_id'],
                    'original_lead_id':               lead.get('original_lead_id', lead['lead_id']),
                    'broker_id':                      neg['broker_id'],
                    'potential_outcome':              int(soft_convert),
                    'potential_conversion_probability': round(would_prob, 4),
                    'potential_revenue':              cf_revenue,
                    'potential_profit':               cf_revenue,
                    'match_quality':                  self.calculate_match_score(lead, neg),
                })

        return assignments, counterfactual_samples

    # =========================================================================
    # ASSIGNMENT GENERATION  (V7.1 base + V7.2 broker churn every 500 leads)
    # =========================================================================

    def generate_assignments(self, leads, brokers, depth=0):
        broker_state = {
            row['broker_id']: {
                'skill':          row['skill_level'],
                'caseload':       row['current_caseload'],
                'recent_success': 0.5,
                'burnout_risk':   row.get('burnout_risk', 0.0),
                'active':         True,
            }
            for _, row in brokers.iterrows()
        }

        # Track any replacement brokers added during churn
        replacement_rows = []

        assignments       = []
        all_counterfactual = []
        leads_with_reentry = []

        print(f"  Processing {len(leads)} leads (depth {depth})...")

        for idx, lead in leads.iterrows():
            if idx % 2000 == 0 and idx > 0:
                print(f"    ... {idx}/{len(leads)} leads processed")

            self.global_time_step += 1

            # Market regime — ONCE per lead (V7.1 fix)
            self.update_market_regime()

            # Natural burnout recovery per lead step
            for k, state in broker_state.items():
                if state.get('active', True):
                    state['burnout_risk'] = max(0.0, state.get('burnout_risk', 0) - 0.001)

            # ── V7.2 graft: broker churn every 500 leads ─────────────────────
            if idx > 0 and idx % 500 == 0:
                broker_state, churned, replacements = self.handle_broker_churn(
                    broker_state, brokers
                )
                if churned:
                    print(f"    Churn at lead {idx}: {len(churned)} retired, "
                          f"{len(replacements)} hired")
                for rep in replacements:
                    replacement_rows.append(rep)
                    # Add to brokers working copy so they can receive leads
                    brokers = pd.concat(
                        [brokers, pd.DataFrame([rep])], ignore_index=True
                    )

            # Build live snapshot of broker pool
            brokers_dynamic = brokers.copy()
            brokers_dynamic['current_caseload'] = brokers_dynamic['broker_id'].map(
                lambda x: broker_state[x]['caseload']
                if broker_state.get(x, {}).get('active', True) else 9999
            )
            active_brokers = brokers_dynamic[
                brokers_dynamic['current_caseload'] < 9999
            ]

            eligible = self.get_eligible_brokers(lead, active_brokers)
            if eligible.empty:
                continue

            lead_assignments, lead_cf = self.simulate_journey(
                lead, eligible, broker_state, depth
            )

            for rec in lead_assignments:
                # 2% log-drop noise
                if random.random() < 0.02:
                    continue
                # 3% timestamp jitter
                if random.random() < 0.03:
                    noise = np.random.randint(-5, 5)
                    if isinstance(rec.get('assignment_date'), datetime):
                        rec['assignment_date'] += timedelta(hours=noise)
                assignments.append(rec)

            all_counterfactual.extend(lead_cf)

            # Lead re-entry (depth-limited)
            if random.random() < self.reentry_rate and depth < self.max_reentry_depth:
                reentry = lead.copy()
                reentry['lead_date'] = lead['lead_date'] + timedelta(
                    days=np.random.randint(7, 30)
                )
                reentry['lead_id']          = f"{lead['lead_id']}_R{len(leads_with_reentry)}"
                reentry['original_lead_id'] = lead['lead_id']
                leads_with_reentry.append(reentry)

        # Process re-entered leads
        if leads_with_reentry and depth < self.max_reentry_depth:
            print(f"  Processing {len(leads_with_reentry)} re-entered leads (depth {depth+1})...")
            reentry_df = pd.DataFrame(leads_with_reentry)
            re_asgn, re_cf = self.generate_assignments(reentry_df, brokers, depth + 1)

            assignments.extend(
                re_asgn.to_dict('records') if isinstance(re_asgn, pd.DataFrame)
                else re_asgn
            )
            all_counterfactual.extend(
                re_cf.to_dict('records') if isinstance(re_cf, pd.DataFrame)
                else re_cf
            )

        return pd.DataFrame(assignments), pd.DataFrame(all_counterfactual)

    # =========================================================================
    # FULL GENERATION PIPELINE
    # =========================================================================

    def generate_all(self, n_brokers=300, n_leads=15000):
        print("=" * 70)
        print("SYNTHETIC DATA GENERATOR V8.0 — PRODUCTION MERGE")
        print("V7.1 causal rigour  +  V7.2 Ontario domain accuracy")
        print("=" * 70)

        print("\nStep 1/4: Generating broker profiles...")
        brokers     = self.generate_brokers(n_brokers)
        new_brokers = brokers[brokers['is_new_broker']]
        ribo_brokers = brokers[brokers['ribo_licensed']]
        print(f"  {len(brokers)} brokers  |  {len(new_brokers)} new  |  "
              f"{len(ribo_brokers)} RIBO-licensed "
              f"({len(ribo_brokers)/len(brokers)*100:.0f}%)")
        print(f"  commission: {brokers['commission_rate'].min():.1%} – "
              f"{brokers['commission_rate'].max():.1%}  |  "
              f"efficiency: {brokers['efficiency'].min():.2f} – "
              f"{brokers['efficiency'].max():.2f}")

        print("\nStep 2/4: Generating lead records...")
        leads = self.generate_leads(n_leads)
        print(f"  {len(leads)} leads generated")
        print(f"  avg quote value: ${leads['quote_value'].mean():.0f}")
        print(f"  multi-product intent: {leads['multi_product_intent'].sum()} "
              f"({leads['multi_product_intent'].mean()*100:.0f}%)")
        print(f"  major claims: {(leads['claims_severity']=='major').sum()} "
              f"({(leads['claims_severity']=='major').mean()*100:.0f}%)")
        missing_pct = (
            leads[['insurance_type', 'language', 'tenure_years']]
            .isna().any(axis=1).mean() * 100
        )
        print(f"  missing data rate: {missing_pct:.1f}%")

        print("\nStep 3/4: Generating assignments (with broker churn)...")
        assignments, counterfactual = self.generate_assignments(leads, brokers)
        print(f"  {len(assignments)} assignment records")
        print(f"  {len(counterfactual)} counterfactual records (evaluation only)")

        print("\nStep 4/4: Building training dataset...")
        broker_cols = [
            'broker_id', 'region', 'expertise_auto', 'expertise_home',
            'expertise_bundle', 'conversion_rate', 'csat_score', 'languages',
            'ribo_licensed', 'ribo_license_years', 'capacity', 'avg_response_time',
            'is_new_broker', 'skill_level', 'reliability', 'commission_rate',
            'cost_per_lead', 'efficiency', 'burnout_risk'
        ]
        historical = (
            assignments
            .merge(leads,          on='lead_id',   how='left')
            .merge(brokers[broker_cols], on='broker_id', how='left')
        )
        for col in ['_score', '_simulated_score']:
            if col in historical.columns:
                historical.drop(columns=[col], inplace=True)

        # ── Summary statistics ────────────────────────────────────────────────
        pos       = assignments[assignments['is_assigned'] == 1]
        converted = pos[pos['converted'] == 1]
        censored  = pos[pos.get('censored', pd.Series(0, index=pos.index)) == 1]

        print("\n" + "=" * 70)
        print("GENERATION COMPLETE — V8.0")
        print("=" * 70)
        print(f"\nDataset statistics:")
        print(f"  Brokers:              {len(brokers):,}")
        print(f"  Leads:                {len(leads):,}")
        print(f"  Total assignments:    {len(assignments):,}")
        print(f"  Positive samples:     {len(pos):,}")
        print(f"  Observed conversions: {len(converted):,}  "
              f"({len(converted)/len(pos)*100:.1f}%)")
        if len(censored):
            print(f"  Censored:             {len(censored):,}")

        pos_prop = pos['propensity_score']
        print(f"\nPropensity scores (positive assignments):")
        print(f"  mean={pos_prop.mean():.4f}  "
              f"min={pos_prop.min():.4f}  max={pos_prop.max():.4f}")
        final_expl = assignments['exploration_rate_at_time'].iloc[-1]
        print(f"  exploration decay: "
              f"{self.initial_exploration_rate:.2%} → {final_expl:.2%}")

        if len(counterfactual):
            cf_conv = counterfactual['potential_outcome'].mean() * 100
            cf_prob = counterfactual['potential_conversion_probability'].mean()
            print(f"\nCounterfactual summary (evaluation only):")
            print(f"  would-have converted: {cf_conv:.1f}%  |  "
                  f"avg probability: {cf_prob:.3f}")

        total_profit = pos['profit'].sum()
        print(f"\nProfit metrics:")
        print(f"  total profit:       ${total_profit:,.2f}")
        print(f"  avg per assignment: ${total_profit/len(pos):.2f}")

        print(f"\nV8.0 feature checklist:")
        print("  [OK] seasonality defined in __init__ (V7.2 crash fixed)")
        print("  [OK] no hidden factor in match score (causal transparency)")
        print("  [OK] propensity_score on every assigned record")
        print("  [OK] action_probabilities logged as JSON")
        print("  [OK] decaying exploration rate (unbiased mixed policy)")
        print("  [OK] uniform negative sampling (no selection-score leakage)")
        print("  [OK] counterfactual CSV restored (5 output files)")
        print("  [OK] update_market_regime called once per lead")
        print("  [OK] event-driven caseload: -1 convert / -0.5 timeout")
        print("  [OK] RIBO as ×1.1 multiplicative (not additive)")
        print("  [OK] claims_severity wired to conversion (major -15%)")
        print("  [OK] multi_product_intent wired to conversion (+15% if bundle)")
        print("  [OK] broker churn + replacement every 500 leads")
        print("  [OK] patience_hours drives max_interactions per lead")
        print("  [OK] original_lead_id preserved for sequence modelling")
        print("  [OK] market_regime logged on every record")

        return brokers, leads, assignments, counterfactual, historical


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    generator = BrokerMatchingDataGenerator(
        random_seed=42,
        observation_window_days=30,
        initial_exploration_rate=0.15,
        exploration_decay=0.01
    )

    brokers, leads, assignments, counterfactual, historical = generator.generate_all(
        n_brokers=300,
        n_leads=15000
    )

    print("\nSaving files...")
    brokers.to_csv('synthetic_brokers_v80.csv',         index=False)
    leads.to_csv('synthetic_leads_v80.csv',             index=False)
    assignments.to_csv('synthetic_assignments_v80.csv', index=False)
    counterfactual.to_csv('synthetic_counterfactual_v80.csv', index=False)
    historical.to_csv('synthetic_historical_v80.csv',   index=False)

    print("\nFiles saved:")
    print("  synthetic_brokers_v80.csv         — broker profiles")
    print("  synthetic_leads_v80.csv           — lead records")
    print("  synthetic_assignments_v80.csv     — full logged policy data")
    print("  synthetic_counterfactual_v80.csv  — evaluation only (ground truth)")
    print("  synthetic_historical_v80.csv      — merged training dataset")

    # ── Quick validation ──────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("VALIDATION")
    print("=" * 70)

    pos = assignments[assignments['is_assigned'] == 1]

    # 1. Propensity scores present
    missing_prop = pos['propensity_score'].isna().sum()
    print(f"\n1. Propensity scores: {missing_prop} missing on positive records "
          f"({'OK' if missing_prop == 0 else 'FAIL'})")

    # 2. No hidden factor — check score range is deterministic-ish
    print("2. Hidden factor: removed from calculate_match_score [OK]")

    # 3. Conversion rate in realistic range
    cr = pos['converted'].mean() * 100
    ok = 5 <= cr <= 25
    print(f"3. Conversion rate: {cr:.1f}%  ({'OK' if ok else 'CHECK — outside 5-25%'})")

    # 4. Price sensitivity visible in data
    med_q = leads['quote_value'].median()
    hi_cr = assignments[
        assignments['lead_id'].isin(leads[leads['quote_value'] > med_q]['lead_id'])
        & (assignments['is_assigned'] == 1)
    ]['converted'].mean() * 100
    lo_cr = assignments[
        assignments['lead_id'].isin(leads[leads['quote_value'] <= med_q]['lead_id'])
        & (assignments['is_assigned'] == 1)
    ]['converted'].mean() * 100
    print(f"4. Price sensitivity: low-price conv={lo_cr:.1f}%  "
          f"high-price conv={hi_cr:.1f}%  "
          f"({'OK' if lo_cr > hi_cr else 'CHECK'})")

    # 5. Counterfactual file non-empty
    print(f"5. Counterfactual records: {len(counterfactual):,} "
          f"({'OK' if len(counterfactual) > 0 else 'FAIL'})")

    # 6. Market regime logged
    has_regime = 'market_regime' in assignments.columns
    print(f"6. Market regime column: {'OK' if has_regime else 'FAIL'}")
    if has_regime:
        print(f"   regime distribution: "
              f"{assignments['market_regime'].value_counts().to_dict()}")

    # 7. RIBO impact
    ribo_ids    = brokers[brokers['ribo_licensed']]['broker_id']
    ribo_conv   = pos[pos['broker_id'].isin(ribo_ids)]['converted'].mean() * 100
    noribo_conv = pos[~pos['broker_id'].isin(ribo_ids)]['converted'].mean() * 100
    print(f"7. RIBO lift: licensed={ribo_conv:.1f}%  "
          f"unlicensed={noribo_conv:.1f}%  "
          f"delta={ribo_conv - noribo_conv:.1f}pp")

    print("\nV8.0 ready for off-policy evaluation and causal model training.")

SYNTHETIC DATA GENERATOR V8.0 — PRODUCTION MERGE
V7.1 causal rigour  +  V7.2 Ontario domain accuracy

Step 1/4: Generating broker profiles...
  300 brokers  |  28 new  |  271 RIBO-licensed (90%)
  commission: 8.0% – 15.0%  |  efficiency: 0.50 – 1.50

Step 2/4: Generating lead records...
  10624 leads generated
  avg quote value: $1573
  multi-product intent: 4335 (41%)
  major claims: 1026 (10%)
  missing data rate: 4.1%

Step 3/4: Generating assignments (with broker churn)...
  Processing 10624 leads (depth 0)...
    Churn at lead 500: 1 retired, 1 hired
    Churn at lead 1000: 1 retired, 1 hired
    ... 2000/10624 leads processed
    Churn at lead 3500: 1 retired, 1 hired
    ... 4000/10624 leads processed
    Churn at lead 4000: 1 retired, 1 hired
    ... 6000/10624 leads processed
    Churn at lead 6500: 2 retired, 2 hired
    ... 8000/10624 leads processed
    ... 10000/10624 leads processed
    Churn at lead 10500: 1 retired, 1 hired
  Processing 850 re-entered leads (depth 1)...